# spore.host — Jupyter Notebook Example

This notebook shows how to use the `spore-host` Python SDK to find, launch, and manage EC2 instances directly from a notebook.

**Prerequisites:**
```sh
pip install spore-host
export SPORE_API_KEY=sk_...    # or AWS credentials already configured
```

In [ ]:
import spore
print(f"spore-host {spore.__version__}")

## 1. Find the right instance

`spore.truffle.find()` searches EC2 instance types using natural language.

In [ ]:
results = spore.truffle.find("amd epyc genoa", region="us-east-1")

# Rich display in Jupyter
import pandas as pd
pd.DataFrame([
    {
        "Instance": r.instance_type,
        "vCPUs": r.vcpus,
        "Memory (GiB)": r.memory_gib,
        "$/hr": f"${r.on_demand_price:.4f}",
        "AZs": ", ".join(r.available_azs),
    }
    for r in results
])

## 2. Check Spot prices

In [ ]:
prices = spore.truffle.spot("c8a.2xlarge", region="us-east-1")
cheapest = min(prices, key=lambda p: p.spot_price)
print(f"Cheapest Spot: {cheapest.availability_zone} @ ${cheapest.spot_price:.4f}/hr ({cheapest.savings_pct:.0f}% savings)")

## 3. Check your account's quota

In [ ]:
quota = spore.truffle.quota("c8a.2xlarge", region="us-east-1")
status = "✅ Can launch" if quota.can_launch else "❌ Quota insufficient"
print(f"{status}: {quota.message}")

## 4. Check running instances

In [ ]:
instances = spore.spawn.list()
if not instances:
    print("No running instances")
else:
    for inst in instances:
        display(inst)   # uses _repr_html_

## 5. Manage an existing instance

Replace `"sim-run-42"` with your instance name.

In [ ]:
inst = spore.spawn.status("sim-run-42")
display(inst)

# Extend TTL if needed
# inst.extend("2h")

# Wake a stopped instance
# inst.start()

# Wait for job to finish (non-blocking poll with callback)
# inst.wait("terminated", poll_interval=60, on_status=lambda i: print(f"{i.state}"))

## 6. Launch and wait (full workflow)

> **Note:** `spawn.launch()` is coming in the next SDK release. Use the CLI for now:
> ```sh
> spawn launch --name my-job --instance-type c8a.2xlarge --ttl 8h --idle-timeout 30m
> ```

Once launch is available, the full pattern will be:

In [ ]:
# Full workflow (launch available in next release)
#
# inst = spore.spawn.launch(
#     "c8a.2xlarge",
#     name="notebook-job",
#     ttl="8h",
#     idle_timeout="30m",
#     on_complete="terminate",
# )
#
# # Poll with progress bar
# from tqdm.notebook import tqdm
# with tqdm(desc="Waiting for job", unit="poll") as pbar:
#     inst.wait("terminated", on_status=lambda i: pbar.set_postfix(state=i.state))
#
# print(f"Done. Instance {inst.name} terminated.")
print("launch() coming soon — use spawn CLI to launch instances")